# Parameter Sensitivity Analysis — Validation Notebook

This notebook validates the 1D, 2D, 3D, and 4D parameter sensitivity analysis implementation.

**What to verify:**
1. **1D**: Line plot with markers, secondary y-axis sample size bars
2. **2D**: 3D surface plot with NaN gaps (not false zeros), `connectgaps=True`
3. **3D**: Interactive plot with dropdown (fix param) + slider (param value)
4. **4D**: Interactive plot with dropdown (fix pair) + slider (pair values)
5. **Report**: Text report with robustness scoring, parameter sensitivity ranking
6. **Dispatcher**: `plot_nd_parameter_analysis()` routes correctly for all dims

In [1]:
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

In [2]:
from utils.core.enums import Ticker, TimeFrame
from datetime import datetime
from feature_extraction.feature_extractor import extract_features_with_forward_returns
from eda.feature_explorer import FeatureExplorer
from feature_selection.base_models.quantile_binning import QuantileBinningModel
from metrics.performance import SortinoRatio
import pandas as pd
import numpy as np

c:\Users\adabla\Trading-Algo\utils\permutation_test\permute_bars.py:38: UserWarning: feature_selection.opt_thresh not found. Using simplified threshold optimization. For full functionality, ensure the opt_thresh module is available.
  warnings.warn(


## Setup: Extract Features with Multi-Param Grid

We use `extract_features_with_forward_returns()` to build a feature set with multiple parameter combinations. The `FeatureExplorer` will parse the standardized column names to detect parameters automatically (no metadata dict needed).

In [3]:
# Extract cumulative_rsi features: lookback x avg_period grid
cumrsi_features, cumrsi_targets = extract_features_with_forward_returns(
    module_name='cumulative_rsi',
    params={
        'lookback': [2, 3, 4, 5],
        'avg_period': [2, 3, 4, 5],
    },
    ticker=Ticker.GC,
    start=datetime(2000, 1, 1),
    end=datetime(2024, 12, 31),
)

# Extract rsi features: lookback grid (1D test)
rsi_features, rsi_targets = extract_features_with_forward_returns(
    module_name='rsi',
    params={'lookback': [5, 7, 10, 14, 20, 30]},
    ticker=Ticker.GC,
    start=datetime(2000, 1, 1),
    end=datetime(2024, 12, 31),
)

# Combine features (align on shared index)
common_idx = cumrsi_features.index.intersection(rsi_features.index)

# Drop ticker column if present before combining
def _drop_ticker(df):
    return df.drop(columns=['ticker'], errors='ignore')

features_df = pd.concat([
    _drop_ticker(cumrsi_features.loc[common_idx]),
    _drop_ticker(rsi_features.loc[common_idx]),
], axis=1)

targets_df = _drop_ticker(cumrsi_targets.loc[common_idx])

# Create FeatureExplorer (no metadata — will parse column names automatically)
explorer = FeatureExplorer(
    features_df=features_df,
    targets_df=targets_df,
)

print(f"Features: {explorer.n_features}")
print(f"Samples: {explorer.n_samples}")
print(f"Date range: {explorer.date_range}")
print(f"Sample columns: {features_df.columns.tolist()[:5]}")

Features: 22
Samples: 6286
Date range: (Timestamp('2000-01-04 00:00:00+0000', tz='UTC'), Timestamp('2024-12-30 00:00:00+0000', tz='UTC'))
Sample columns: ['cumulative_rsi_signal_D_avgPeriod_2_lookback_2', 'cumulative_rsi_signal_D_avgPeriod_3_lookback_2', 'cumulative_rsi_signal_D_avgPeriod_4_lookback_2', 'cumulative_rsi_signal_D_avgPeriod_5_lookback_2', 'cumulative_rsi_signal_D_avgPeriod_2_lookback_3']


In [4]:
# Shared model and metric
model = QuantileBinningModel(n_bins=5, selection_metric='sortino')
metric = SortinoRatio(annualization_factor=252)

## Verify: Available parameter groups

Check that the explorer correctly detected parameterized features.

In [5]:
param_groups = explorer.get_parameterized_features()
print("Parameter groups found:")
for (module, param), features in sorted(param_groups.items()):
    values = sorted(set(v for v, _ in features))
    print(f"  {module}.{param}: {len(features)} features, values={values}")

Parameter groups found:
  cumulative_rsi.avgPeriod: 16 features, values=[2, 3, 4, 5]
  cumulative_rsi.lookback: 16 features, values=[2, 3, 4, 5]
  rsi.lookback: 6 features, values=[5, 7, 10, 14, 20, 30]


---
## Test 1: 1D Parameter Sensitivity (RSI lookback)

**Expected**: Line chart with markers (Sortino on left y-axis), sample size bars (right y-axis).  
**Verify**: X-axis shows lookback values [5,7,10,14,20,30], line is smooth, bars are gray/transparent.

In [6]:
# Direct call
df_1d, fig_1d = explorer.plot_parameter_sensitivity(
    module_name='rsi',
    param_name='lookback',
    feature_name='signal',
    target_col='log_return',
    base_model=model,
    metric=metric
)
print("1D Results:")
df_1d

[DEBUG] analyze_parameter: 'rsi_signal_D_lookback_5' samples=6286 (feature NaNs=0, target NaNs=0)
[DEBUG] analyze_parameter: 'rsi_signal_D_lookback_7' samples=6286 (feature NaNs=0, target NaNs=0)
[DEBUG] analyze_parameter: 'rsi_signal_D_lookback_10' samples=6286 (feature NaNs=0, target NaNs=0)
[DEBUG] analyze_parameter: 'rsi_signal_D_lookback_14' samples=6286 (feature NaNs=0, target NaNs=0)
[DEBUG] analyze_parameter: 'rsi_signal_D_lookback_20' samples=6286 (feature NaNs=0, target NaNs=0)
[DEBUG] analyze_parameter: 'rsi_signal_D_lookback_30' samples=6286 (feature NaNs=0, target NaNs=0)


1D Results:


,param1_value,sortino,mean,std,max_drawdown,n_samples
0,5,0.526841,0.000241,0.010342,-0.322234,1257
1,7,1.203049,0.000523,0.009811,-0.221545,1257
2,10,1.145925,0.000532,0.010051,-0.262863,1257
3,14,0.509093,0.000287,0.010691,-0.238674,1257
4,20,0.540186,0.000306,0.010700,-0.239388,1257
5,30,0.547068,0.000292,0.010728,-0.281540,1257


In [7]:
# Via dispatcher (should produce identical result)
df_1d_dispatch, fig_1d_dispatch = explorer.plot_nd_parameter_analysis(
    module_name='rsi',
    param_names=['lookback'],
    feature_name='signal',
    target_col='log_return',
    base_model=model,
    metric=metric
)
print("Dispatcher 1D matches direct call:", df_1d.shape == df_1d_dispatch.shape)

[DEBUG] analyze_parameter: 'rsi_signal_D_lookback_5' samples=6286 (feature NaNs=0, target NaNs=0)
[DEBUG] analyze_parameter: 'rsi_signal_D_lookback_7' samples=6286 (feature NaNs=0, target NaNs=0)
[DEBUG] analyze_parameter: 'rsi_signal_D_lookback_10' samples=6286 (feature NaNs=0, target NaNs=0)
[DEBUG] analyze_parameter: 'rsi_signal_D_lookback_14' samples=6286 (feature NaNs=0, target NaNs=0)
[DEBUG] analyze_parameter: 'rsi_signal_D_lookback_20' samples=6286 (feature NaNs=0, target NaNs=0)
[DEBUG] analyze_parameter: 'rsi_signal_D_lookback_30' samples=6286 (feature NaNs=0, target NaNs=0)


Dispatcher 1D matches direct call: True


---
## Test 2: 2D Parameter Surface (cumulative_rsi: lookback x avgPeriod)

**Expected**: 3D surface plot with Viridis colorscale.  
**Verify**:
- NaN gaps are NOT filled with zeros (hover over cells to check)
- Surface has `connectgaps=True` for visual continuity
- Axes labeled correctly (lookback on y, avgPeriod on x, Sortino on z)

In [8]:
df_2d, fig_2d = explorer.plot_2d_parameter_surface(
    module_name='cumulative_rsi',
    param1_name='lookback',
    param2_name='avgPeriod',
    feature_name='signal',
    target_col='log_return',
    base_model=model,
    metric=metric,
    plot_type='surface'
)
print("2D Results shape:", df_2d.shape)
df_2d

Analyzing 2D parameters with 16 parameter combinations
  Processing parameter combination: (4, 4) with 1 features
    Processing feature: cumulative_rsi_signal_D_avgPeriod_4_lookback_4
      [DEBUG] aligned samples=6286 (feature NaNs=0, target NaNs=0)
      Completed
  Processing parameter combination: (2, 4) with 1 features
    Processing feature: cumulative_rsi_signal_D_avgPeriod_4_lookback_2
      [DEBUG] aligned samples=6286 (feature NaNs=0, target NaNs=0)
      Completed
  Processing parameter combination: (5, 5) with 1 features
    Processing feature: cumulative_rsi_signal_D_avgPeriod_5_lookback_5
      [DEBUG] aligned samples=6286 (feature NaNs=0, target NaNs=0)
      Completed
  Processing parameter combination: (3, 4) with 1 features
    Processing feature: cumulative_rsi_signal_D_avgPeriod_4_lookback_3
      [DEBUG] aligned samples=6286 (feature NaNs=0, target NaNs=0)
      Completed
  Processing parameter combination: (4, 3) with 1 features
    Processing feature: cumulative

2D Results shape: (16, 7)


,param1_value,param2_value,sortino,mean,std,max_drawdown,n_samples
0,2,2,1.129622,0.000465,0.009564,-0.250428,1257
1,2,3,1.164565,0.000529,0.010085,-0.151649,1257
2,2,4,0.796658,0.000379,0.010055,-0.282340,1257
3,2,5,1.153086,0.000496,0.009787,-0.302702,1257
4,3,2,1.124673,0.000519,0.010293,-0.232996,1257
5,3,3,0.639060,0.000305,0.010220,-0.349893,1257
6,3,4,0.519537,0.000244,0.010026,-0.184093,1258
7,3,5,0.734231,0.000337,0.010049,-0.227732,1257
8,4,2,1.026015,0.000464,0.010024,-0.221948,1257
9,4,3,0.582827,0.000276,0.010275,-0.174992,1258


In [9]:
# Also test heatmap mode
df_2d_hm, fig_2d_hm = explorer.plot_2d_parameter_surface(
    module_name='cumulative_rsi',
    param1_name='lookback',
    param2_name='avgPeriod',
    feature_name='signal',
    target_col='log_return',
    base_model=model,
    metric=metric,
    plot_type='heatmap'
)

Analyzing 2D parameters with 16 parameter combinations
  Processing parameter combination: (4, 4) with 1 features
    Processing feature: cumulative_rsi_signal_D_avgPeriod_4_lookback_4
      [DEBUG] aligned samples=6286 (feature NaNs=0, target NaNs=0)
      Completed
  Processing parameter combination: (2, 4) with 1 features
    Processing feature: cumulative_rsi_signal_D_avgPeriod_4_lookback_2
      [DEBUG] aligned samples=6286 (feature NaNs=0, target NaNs=0)
      Completed
  Processing parameter combination: (5, 5) with 1 features
    Processing feature: cumulative_rsi_signal_D_avgPeriod_5_lookback_5
      [DEBUG] aligned samples=6286 (feature NaNs=0, target NaNs=0)
      Completed
  Processing parameter combination: (3, 4) with 1 features
    Processing feature: cumulative_rsi_signal_D_avgPeriod_4_lookback_3
      [DEBUG] aligned samples=6286 (feature NaNs=0, target NaNs=0)
      Completed
  Processing parameter combination: (4, 3) with 1 features
    Processing feature: cumulative

In [10]:
# Via dispatcher
df_2d_dispatch, fig_2d_dispatch = explorer.plot_nd_parameter_analysis(
    module_name='cumulative_rsi',
    param_names=['lookback', 'avgPeriod'],
    feature_name='signal',
    target_col='log_return',
    base_model=model,
    metric=metric
)
print("Dispatcher 2D matches direct call:", df_2d.shape == df_2d_dispatch.shape)

Analyzing 2D parameters with 16 parameter combinations
  Processing parameter combination: (4, 4) with 1 features
    Processing feature: cumulative_rsi_signal_D_avgPeriod_4_lookback_4
      [DEBUG] aligned samples=6286 (feature NaNs=0, target NaNs=0)
      Completed
  Processing parameter combination: (2, 4) with 1 features
    Processing feature: cumulative_rsi_signal_D_avgPeriod_4_lookback_2
      [DEBUG] aligned samples=6286 (feature NaNs=0, target NaNs=0)
      Completed
  Processing parameter combination: (5, 5) with 1 features
    Processing feature: cumulative_rsi_signal_D_avgPeriod_5_lookback_5
      [DEBUG] aligned samples=6286 (feature NaNs=0, target NaNs=0)
      Completed
  Processing parameter combination: (3, 4) with 1 features
    Processing feature: cumulative_rsi_signal_D_avgPeriod_4_lookback_3
      [DEBUG] aligned samples=6286 (feature NaNs=0, target NaNs=0)
      Completed
  Processing parameter combination: (4, 3) with 1 features
    Processing feature: cumulative

Dispatcher 2D matches direct call: True


### 2D fillna bug verification

Create a sparse grid and verify NaN is preserved (not filled with 0).

In [11]:
from metrics.plotting.parameter_plots import plot_2d_parameter_surface as plot_2d_pure

# Sparse grid: (3, 20) is missing
sparse_df = pd.DataFrame({
    'param1_value': [1, 1, 2, 2, 3],
    'param2_value': [10, 20, 10, 20, 10],
    'sortino':      [1.5, 2.0, 1.8, 2.2, 1.6],
})

fig_sparse = plot_2d_pure(sparse_df, 'p1', 'p2', 'sortino', show_plot=False)
z_vals = np.array(fig_sparse.data[0].z)
print("Z values (NaN = gap, not 0):")
print(z_vals)
print(f"\nNaN count: {np.isnan(z_vals).sum()} (expected: 1)")
print(f"Zero count: {(z_vals == 0.0).sum()} (expected: 0 if fix worked)")
fig_sparse.show()

Z values (NaN = gap, not 0):
[[1.5 2. ]
 [1.8 2.2]
 [1.6 nan]]

NaN count: 1 (expected: 1)
Zero count: 0 (expected: 0 if fix worked)


---
## Test 3: 3D Interactive Visualization

We need a module with 3+ parameters. We'll use `_extract_parameter_grid` + `analyze_nd_parameters` + `plot_3d_parameter_interactive` directly, since cumulative_rsi only has 2 params.

**For this test, we use synthetic data** to guarantee a 3-param grid.

**Expected**:
- Dropdown with 3 options: "Fix param1", "Fix param2", "Fix param3"
- Slider steps through values of the fixed parameter
- Surface updates to show remaining 2 params vs metric
- Axes update dynamically

In [12]:
from metrics.plotting.parameter_plots import plot_3d_parameter_interactive

# Synthetic 3D results DataFrame
rows_3d = []
rng = np.random.RandomState(42)
for p1 in [5, 10, 15, 20]:
    for p2 in [30, 50, 70]:
        for p3 in [1, 2, 3]:
            rows_3d.append({
                'param1_value': p1,
                'param2_value': p2,
                'param3_value': p3,
                'sortino': 1.0 + (p1/20) + (p2/100) - (p3*0.1) + rng.randn()*0.2,
                'n_samples': 200 + rng.randint(-50, 50),
                'max_drawdown': -0.05 - rng.rand()*0.1,
            })

df_3d = pd.DataFrame(rows_3d)
print(f"3D DataFrame: {df_3d.shape}")
print(f"Param1 values: {sorted(df_3d.param1_value.unique())}")
print(f"Param2 values: {sorted(df_3d.param2_value.unique())}")
print(f"Param3 values: {sorted(df_3d.param3_value.unique())}")

fig_3d = plot_3d_parameter_interactive(
    df=df_3d,
    param_names=['lookback', 'threshold', 'smoothing'],
    metric='sortino',
    show_plot=True,
    plot_type='surface'
)

3D DataFrame: (36, 6)
Param1 values: [np.int64(5), np.int64(10), np.int64(15), np.int64(20)]
Param2 values: [np.int64(30), np.int64(50), np.int64(70)]
Param3 values: [np.int64(1), np.int64(2), np.int64(3)]


In [13]:
# Also test heatmap mode for 3D
fig_3d_hm = plot_3d_parameter_interactive(
    df=df_3d,
    param_names=['lookback', 'threshold', 'smoothing'],
    metric='sortino',
    show_plot=True,
    plot_type='heatmap'
)

---
## Test 4: 4D Interactive Visualization

**Expected**:
- Dropdown with C(4,2)=6 options: "Fix paramA & paramB"
- Slider labeled "paramA=X, paramB=Y" stepping through all combos of fixed params
- Surface shows remaining 2 params vs metric
- Axes update dynamically

In [14]:
from metrics.plotting.parameter_plots import plot_4d_parameter_interactive

# Synthetic 4D results DataFrame
rows_4d = []
rng = np.random.RandomState(123)
for p1 in [5, 10, 15]:
    for p2 in [30, 50, 70]:
        for p3 in [1, 2, 3]:
            for p4 in [0.5, 1.0]:
                rows_4d.append({
                    'param1_value': p1,
                    'param2_value': p2,
                    'param3_value': p3,
                    'param4_value': p4,
                    'sortino': 1.5 + (p1/15) + (p2/100) - (p3*0.15) + p4*0.3 + rng.randn()*0.15,
                    'n_samples': 250 + rng.randint(-30, 30),
                    'max_drawdown': -0.04 - rng.rand()*0.08,
                })

df_4d = pd.DataFrame(rows_4d)
print(f"4D DataFrame: {df_4d.shape}")
print(f"Param1 values: {sorted(df_4d.param1_value.unique())}")
print(f"Param2 values: {sorted(df_4d.param2_value.unique())}")
print(f"Param3 values: {sorted(df_4d.param3_value.unique())}")
print(f"Param4 values: {sorted(df_4d.param4_value.unique())}")

fig_4d = plot_4d_parameter_interactive(
    df=df_4d,
    param_names=['lookback', 'threshold', 'smoothing', 'decay'],
    metric='sortino',
    show_plot=True,
    plot_type='surface'
)

4D DataFrame: (54, 7)
Param1 values: [np.int64(5), np.int64(10), np.int64(15)]
Param2 values: [np.int64(30), np.int64(50), np.int64(70)]
Param3 values: [np.int64(1), np.int64(2), np.int64(3)]
Param4 values: [np.float64(0.5), np.float64(1.0)]


In [15]:
# Test 4D contour mode
fig_4d_c = plot_4d_parameter_interactive(
    df=df_4d,
    param_names=['lookback', 'threshold', 'smoothing', 'decay'],
    metric='sortino',
    show_plot=True,
    plot_type='contour'
)

---
## Test 5: Robustness Metrics

**Verify**:
- Variance score: `max(0, 10 - cv * 20)`
- Consistency score: `pct_above_threshold * 10`
- Risk score: `max(0, 10 - |min(max_drawdown)| * 50)`
- Overall: average of 3 components
- Rating matches thresholds (8+ HIGHLY ROBUST, 6+ ROBUST, etc.)
- Parameter sensitivity sums to 1.0

In [16]:
from eda.parameter_analysis import ParameterAnalyzer

analyzer = ParameterAnalyzer(explorer.features_df, explorer.targets_df)

# Use the 3D synthetic data
robustness = analyzer.compute_robustness_metrics(
    results_df=df_3d,
    metric_col='sortino',
    metric_threshold=1.0
)

print("ROBUSTNESS METRICS (3D synthetic data)")
print("=" * 50)
print(f"Variance Score:    {robustness['variance_score']:.2f} / 10")
print(f"Consistency Score: {robustness['consistency_score']:.2f} / 10")
print(f"Risk Score:        {robustness['risk_score']:.2f} / 10")
print(f"Overall Score:     {robustness['overall_score']:.2f} / 10")
print(f"Rating:            {robustness['rating']}")
print()
print("Statistics:")
for k, v in robustness['statistics'].items():
    print(f"  {k}: {v}")
print()
print("Parameter Sensitivity:")
sensitivity = robustness['parameter_sensitivity']
for param, pct in sorted(sensitivity.items(), key=lambda x: x[1], reverse=True):
    print(f"  {param}: {pct:.2%}")
print(f"  Sum: {sum(sensitivity.values()):.4f} (should be 1.0)")

ROBUSTNESS METRICS (3D synthetic data)
Variance Score:    5.35 / 10
Consistency Score: 9.72 / 10
Risk Score:        2.54 / 10
Overall Score:     5.87 / 10
Rating:            MODERATELY ROBUST

Statistics:
  mean: 1.919712893395194
  median: 1.9235901806895148
  std: 0.44628120496388946
  min: 0.8478074220510744
  max: 2.756841637530788
  cv: 0.23247289034695126
  n_configurations: 36
  pct_above_threshold: 0.9722222222222222

Parameter Sensitivity:
  param1: 77.29%
  param2: 19.37%
  param3: 3.34%
  Sum: 1.0000 (should be 1.0)


---
## Test 6: Full Report Generation (with real data)

**Verify**:
- Report text has all sections: PARAMETER SPACE, PERFORMANCE SUMMARY, ROBUSTNESS ANALYSIS, PARAMETER SENSITIVITY RANKING, RECOMMENDATIONS
- Return dict has all keys: results_df, summary_stats, robustness_scores, figures, parameter_sensitivity, report_text, report_path
- File export works when export_path is given

In [17]:
# Generate report for cumulative_rsi (2 params: lookback, avgPeriod)
report = explorer.generate_parameter_sensitivity_report(
    module_name='cumulative_rsi',
    param_names=['lookback', 'avgPeriod'],
    target_col='log_return',
    metric=metric,
    base_model=model,
    metric_threshold=1.0,
    verbose=True
)

Analyzing 2D parameters with 16 parameter combinations
  Processing parameter combination: (4, 4) with 1 features
    Processing feature: cumulative_rsi_signal_D_avgPeriod_4_lookback_4
      [DEBUG] aligned samples=6286 (feature NaNs=0, target NaNs=0)
      Completed
  Processing parameter combination: (2, 4) with 1 features
    Processing feature: cumulative_rsi_signal_D_avgPeriod_4_lookback_2
      [DEBUG] aligned samples=6286 (feature NaNs=0, target NaNs=0)
      Completed
  Processing parameter combination: (5, 5) with 1 features
    Processing feature: cumulative_rsi_signal_D_avgPeriod_5_lookback_5
      [DEBUG] aligned samples=6286 (feature NaNs=0, target NaNs=0)
      Completed
  Processing parameter combination: (3, 4) with 1 features
    Processing feature: cumulative_rsi_signal_D_avgPeriod_4_lookback_3
      [DEBUG] aligned samples=6286 (feature NaNs=0, target NaNs=0)
      Completed
  Processing parameter combination: (4, 3) with 1 features
    Processing feature: cumulative

In [18]:
# Verify return dict structure
print("Report keys:", list(report.keys()))
print()
print(f"Results DF shape: {report['results_df'].shape}")
print(f"Robustness rating: {report['robustness_scores']['rating']}")
print(f"Overall score: {report['robustness_scores']['overall_score']:.2f}")
print(f"Figures count: {len(report['figures'])}")
print(f"Parameter sensitivity: {report['parameter_sensitivity']}")
print(f"Report path: {report['report_path']}")

Report keys: ['results_df', 'summary_stats', 'robustness_scores', 'figures', 'parameter_sensitivity', 'report_text', 'report_path']

Results DF shape: (16, 7)
Robustness rating: FRAGILE
Overall score: 2.40
Figures count: 1
Parameter sensitivity: {'param1': np.float64(0.5844363976568667), 'param2': np.float64(0.4155636023431333)}
Report path: None


In [19]:
# Test auto-detection: pass param_names=None
report_auto = explorer.generate_parameter_sensitivity_report(
    module_name='cumulative_rsi',
    param_names=None,  # auto-detect all params
    metric=metric,
    base_model=model,
    verbose=True
)

Auto-detected parameters for 'cumulative_rsi': ['avgPeriod', 'lookback']
Analyzing 2D parameters with 16 parameter combinations
  Processing parameter combination: (4, 4) with 1 features
    Processing feature: cumulative_rsi_signal_D_avgPeriod_4_lookback_4
      [DEBUG] aligned samples=6286 (feature NaNs=0, target NaNs=0)
      Completed
  Processing parameter combination: (2, 4) with 1 features
    Processing feature: cumulative_rsi_signal_D_avgPeriod_2_lookback_4
      [DEBUG] aligned samples=6286 (feature NaNs=0, target NaNs=0)
      Completed
  Processing parameter combination: (5, 5) with 1 features
    Processing feature: cumulative_rsi_signal_D_avgPeriod_5_lookback_5
      [DEBUG] aligned samples=6286 (feature NaNs=0, target NaNs=0)
      Completed
  Processing parameter combination: (3, 4) with 1 features
    Processing feature: cumulative_rsi_signal_D_avgPeriod_3_lookback_4
      [DEBUG] aligned samples=6286 (feature NaNs=0, target NaNs=0)
      Completed
  Processing paramet

In [20]:
# Test file export
import tempfile, os

with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as f:
    export_path = f.name

report_export = explorer.generate_parameter_sensitivity_report(
    module_name='rsi',
    param_names=['lookback'],
    metric=metric,
    base_model=model,
    export_path=export_path,
    verbose=False
)

print(f"Exported to: {report_export['report_path']}")
print(f"File exists: {os.path.exists(export_path)}")
print(f"File size: {os.path.getsize(export_path)} bytes")
print()
with open(export_path) as f:
    print(f.read())

os.unlink(export_path)

[DEBUG] analyze_parameter: 'rsi_signal_D_lookback_5' samples=6286 (feature NaNs=0, target NaNs=0)
[DEBUG] analyze_parameter: 'rsi_signal_D_lookback_7' samples=6286 (feature NaNs=0, target NaNs=0)
[DEBUG] analyze_parameter: 'rsi_signal_D_lookback_10' samples=6286 (feature NaNs=0, target NaNs=0)
[DEBUG] analyze_parameter: 'rsi_signal_D_lookback_14' samples=6286 (feature NaNs=0, target NaNs=0)
[DEBUG] analyze_parameter: 'rsi_signal_D_lookback_20' samples=6286 (feature NaNs=0, target NaNs=0)
[DEBUG] analyze_parameter: 'rsi_signal_D_lookback_30' samples=6286 (feature NaNs=0, target NaNs=0)
Exported to: C:\Users\adabla\AppData\Local\Temp\tmp6j8dzz5x.txt
File exists: True
File size: 1213 bytes

PARAMETER SENSITIVITY REPORT: RSI

PARAMETER SPACE
----------------------------------------
  lookback: [np.int64(5), np.int64(7), np.int64(10), np.int64(14), np.int64(20), np.int64(30)]
  Total configurations: 6

PERFORMANCE SUMMARY
----------------------------------------
  Metric: sortino
  Mean:   

---
## Test 7: _extract_parameter_grid validation

Verify the shared grid extraction works correctly for different dimensions.

In [21]:
# 1-param grid
grid_1d = explorer._extract_parameter_grid('rsi', ['lookback'])
print(f"1D grid: {len(grid_1d)} entries (expected: 6 lookback values)")
for k, v in sorted(grid_1d.items()):
    print(f"  {k} -> {len(v)} features")

print()

# 2-param grid
grid_2d = explorer._extract_parameter_grid('cumulative_rsi', ['lookback', 'avgPeriod'])
print(f"2D grid: {len(grid_2d)} entries (expected: 4x4=16)")
for k, v in sorted(grid_2d.items()):
    print(f"  {k} -> {len(v)} features: {v[:2]}...")

1D grid: 6 entries (expected: 6 lookback values)
  (5,) -> 1 features
  (7,) -> 1 features
  (10,) -> 1 features
  (14,) -> 1 features
  (20,) -> 1 features
  (30,) -> 1 features

2D grid: 16 entries (expected: 4x4=16)
  (2, 2) -> 1 features: ['cumulative_rsi_signal_D_avgPeriod_2_lookback_2']...
  (2, 3) -> 1 features: ['cumulative_rsi_signal_D_avgPeriod_3_lookback_2']...
  (2, 4) -> 1 features: ['cumulative_rsi_signal_D_avgPeriod_4_lookback_2']...
  (2, 5) -> 1 features: ['cumulative_rsi_signal_D_avgPeriod_5_lookback_2']...
  (3, 2) -> 1 features: ['cumulative_rsi_signal_D_avgPeriod_2_lookback_3']...
  (3, 3) -> 1 features: ['cumulative_rsi_signal_D_avgPeriod_3_lookback_3']...
  (3, 4) -> 1 features: ['cumulative_rsi_signal_D_avgPeriod_4_lookback_3']...
  (3, 5) -> 1 features: ['cumulative_rsi_signal_D_avgPeriod_5_lookback_3']...
  (4, 2) -> 1 features: ['cumulative_rsi_signal_D_avgPeriod_2_lookback_4']...
  (4, 3) -> 1 features: ['cumulative_rsi_signal_D_avgPeriod_3_lookback_4']...
 

---
## Summary Checklist

After running all cells, verify:

| # | Test | What to Check |
|---|------|---------------|
| 1 | 1D line plot | Markers + line on left y-axis, gray bars on right y-axis |
| 2 | 2D surface | 3D surface with correct axes; NaN preserved (not 0) |
| 3 | 3D interactive | Dropdown toggles which param is fixed; slider steps through values |
| 4 | 4D interactive | Dropdown shows 6 pair combos; slider shows "paramA=X, paramB=Y" |
| 5 | Robustness | Scores 0-10, rating matches thresholds, sensitivity sums to 1 |
| 6 | Report | All sections present, dict has all keys, file export works |
| 7 | Grid extraction | Correct counts for 1D and 2D grids |